In [ ]:
from functions import (load_json,
                       save_json,
                       past_example_components,
                       get_job_details,
                       get_prompts,
                       get_full_prompt,
                       split_llm_outputs,
                       make_doc,
                       save_to_folder)
from gemini_chatbot import ChatBot

# Load details from past applications from json file, parse document components
past_examples = load_json('input/job_application_examples.json')
doc_components = past_example_components(past_examples)

# Get job posting details
job_details = get_job_details()

# Get prompt templates from .txt file
prompts = get_prompts(job_details)

# Generate text for document components
chatbot = ChatBot()
raw_llm_outputs = {}
for key in [*doc_components[0].keys()]:
    raw_llm_outputs[key] = chatbot.send(get_full_prompt(key, prompts, doc_components))
    print(f'\rFinished {key}{' '*20}', end='')

# Extract feedback on how much new text was composed by AI, for QA
llm_outputs, llm_report = split_llm_outputs(raw_llm_outputs)

# Create resume and cover letter by inserting generated text into templates
resume = make_doc('resume', job_details, llm_outputs)
coverletter = make_doc('coverletter', job_details, llm_outputs)

# Write documents to files, update past examples json
past_examples_add = save_to_folder(job_details, resume, coverletter, llm_report)
past_examples += past_examples_add
save_json('input/job_application_examples.json', past_examples)